US STATES HEALTH ANALYSIS

In [105]:
import re
import pandas as pd
import numpy as np
import matplotlib as mpl
%matplotlib inline
from functools import reduce

Creating dataframes for all the csv files to be Analysed and merged after cleaning

In [106]:
pd.set_option("display.max.columns", None)

In [107]:
diverse_states_df = pd.read_csv('diverse_states.csv')
expensive_states_df = pd.read_csv('expensive_states_healthcare.csv')
health_care_costs_by_state_df = pd.read_csv('healthcare_costs_share_of_income_by_state.csv')
healthcare_spending_per_person_df = pd.read_csv('healthcare_spending_per_person_by_state.csv')
healthiest_states_df = pd.read_csv('healthiest_states.csv')
life_expectancy_by_state_df = pd.read_csv('life_expectancy_by_state.csv')
obesity_rate_df = pd.read_csv('obesity_rates_in_US_States.csv')

In [108]:
obesity_rate_df.rename(columns={'State or Territory': 'State'}, inplace=True) # Changed column name for merging

Creating a sorted list of dataframes to prepare for merging

In [109]:
dfs = [diverse_states_df, expensive_states_df, health_care_costs_by_state_df, healthcare_spending_per_person_df,
       healthiest_states_df, life_expectancy_by_state_df, obesity_rate_df]

Dropping duplicate column on the dataframes to prevent suffix errors while merging

In [110]:
dfs = [df.drop('Rank', axis=1) for df in dfs]   

Merging on the State column

In [111]:
df_merged = reduce(lambda  left, right: pd.merge(left, right, on=['State'], how='outer', suffixes=('_left', '_right')), dfs)

Viewing merged dataframe

In [112]:
df_merged.head()

,State,Diversity Index (%),Healthcare Cost Index (U.S. = 100),Health Care as % of Household Income,Per-Capita Health Care Spending,Overall Score 2025,Life Expectancy (Years),Adult Obesity Rate (2024)
0,Alabama,53.1,85.6,7.32%,"$7,980",-0.75,73.8,38.9%
1,Alaska,62.8,130.7,10.08%,"$14,044",-0.13,75.8,34.0%
2,Arizona,61.5,98.8,6.81%,"$8,556",-0.06,76.7,33.3%
3,Arkansas,49.8,85.8,7.67%,"$8,562",-0.83,73.9,38.9%
4,California,69.7,111.1,5.64%,"$11,054",0.10,79.3,29.1%


In [113]:
df_merged.tail()

,State,Diversity Index (%),Healthcare Cost Index (U.S. = 100),Health Care as % of Household Income,Per-Capita Health Care Spending,Overall Score 2025,Life Expectancy (Years),Adult Obesity Rate (2024)
52,"Washington, D.C.",67.2,NaN,NaN,NaN,NaN,NaN,NaN
53,West Virginia,20.2,89.5,9.14%,"$12,055",-0.73,72.2,41.4%
54,Wisconsin,37.0,105.3,6.76%,"$10,079",0.16,78.1,37.4%
55,Wyoming,32.4,97.0,7.84%,"$9,640",-0.04,76.8,32.5%
56,🇺🇸 U.S. State and Territory Average,NaN,NaN,NaN,NaN,NaN,NaN,34.1%


In [114]:
df_merged.rename(columns={'Overall Score 2025': 'Health Score', 'Adult Obesity Rate (2024)': 'Adult Obesity (%)', 
                          'Per-Capita Health Care Spending':'Per-Capita Health Care Spending ($)'},
                 inplace=True) # Changed column names to properly reflect column data

In [115]:
df_merged.head()

,State,Diversity Index (%),Healthcare Cost Index (U.S. = 100),Health Care as % of Household Income,Per-Capita Health Care Spending ($),Health Score,Life Expectancy (Years),Adult Obesity (%)
0,Alabama,53.1,85.6,7.32%,"$7,980",-0.75,73.8,38.9%
1,Alaska,62.8,130.7,10.08%,"$14,044",-0.13,75.8,34.0%
2,Arizona,61.5,98.8,6.81%,"$8,556",-0.06,76.7,33.3%
3,Arkansas,49.8,85.8,7.67%,"$8,562",-0.83,73.9,38.9%
4,California,69.7,111.1,5.64%,"$11,054",0.10,79.3,29.1%


Removing %, $ and commas using regex from the column data for Visualization

In [116]:
df_merged.replace(r"(^\$)|(%$)|(\,)", '', regex=True, inplace=True)

,State,Diversity Index (%),Healthcare Cost Index (U.S. = 100),Health Care as % of Household Income,Per-Capita Health Care Spending ($),Health Score,Life Expectancy (Years),Adult Obesity (%)
0,Alabama,53.1,85.6,7.32,7980,-0.75,73.8,38.9
1,Alaska,62.8,130.7,10.08,14044,-0.13,75.8,34.0
2,Arizona,61.5,98.8,6.81,8556,-0.06,76.7,33.3
3,Arkansas,49.8,85.8,7.67,8562,-0.83,73.9,38.9
4,California,69.7,111.1,5.64,11054,0.10,79.3,29.1
5,Colorado,52.3,103.9,6.05,8871,0.51,78.5,25.0
6,Connecticut,55.7,108.9,6.89,10639,0.68,79.4,32.0
7,Delaware,59.6,101.6,6.66,11987,0.23,76.5,36.6
8,District of Columbia,NaN,118.7,NaN,13865,NaN,76.6,25.5
9,Florida,64.1,101.4,6.05,9545,-0.05,77.9,29.6


Merging and grouping split Tennessee Data

In [117]:
df_merged["State"] = df_merged["State"].str.replace("*", "", regex=False)
df = df_merged.groupby("State", as_index=False).first()

In [118]:
df.head()

,State,Diversity Index (%),Healthcare Cost Index (U.S. = 100),Health Care as % of Household Income,Per-Capita Health Care Spending ($),Health Score,Life Expectancy (Years),Adult Obesity (%)
0,Alabama,53.1,85.6,7.32,7980,-0.75,73.8,38.9
1,Alaska,62.8,130.7,10.08,14044,-0.13,75.8,34.0
2,Arizona,61.5,98.8,6.81,8556,-0.06,76.7,33.3
3,Arkansas,49.8,85.8,7.67,8562,-0.83,73.9,38.9
4,California,69.7,111.1,5.64,11054,0.10,79.3,29.1


Final compiled CSV output

In [119]:
df.to_csv('final_data.csv', sep=',', index=False, encoding='utf-8', na_rep='NaN')

Understanding the DataFrame

In [120]:
df.columns

Index(['State', 'Diversity Index (%)', 'Healthcare Cost Index (U.S. = 100)',
       'Health Care as % of Household Income',
       'Per-Capita Health Care Spending ($)', 'Health Score',
       'Life Expectancy (Years)', 'Adult Obesity (%)'],
      dtype='str')

In [121]:
df.dtypes

State                                       str
Diversity Index (%)                     float64
Healthcare Cost Index (U.S. = 100)      float64
Health Care as % of Household Income        str
Per-Capita Health Care Spending ($)         str
Health Score                            float64
Life Expectancy (Years)                 float64
Adult Obesity (%)                           str
dtype: object

Convert str Columns to float

In [122]:
df['Health Care as % of Household Income'] = df['Health Care as % of Household Income'].astype('float64')
df['Per-Capita Health Care Spending ($)'] = df['Per-Capita Health Care Spending ($)'].astype('float64')
df['Adult Obesity (%)'] = df['Adult Obesity (%)'].astype('float64')

In [123]:
df.dtypes

State                                       str
Diversity Index (%)                     float64
Healthcare Cost Index (U.S. = 100)      float64
Health Care as % of Household Income    float64
Per-Capita Health Care Spending ($)     float64
Health Score                            float64
Life Expectancy (Years)                 float64
Adult Obesity (%)                       float64
dtype: object

In [124]:
df.describe()

,Diversity Index (%),Healthcare Cost Index (U.S. = 100),Health Care as % of Household Income,Per-Capita Health Care Spending ($),Health Score,Life Expectancy (Years),Adult Obesity (%)
count,51.000000,51.000000,50.000000,51.000000,50.000000,51.000000,55.000000
mean,49.398039,102.358824,7.290400,9915.705882,0.056800,76.878431,34.114545
std,14.598445,10.398580,1.133749,1531.949468,0.479496,1.998130,3.878785
min,18.500000,85.100000,5.110000,7233.000000,-0.940000,72.200000,25.000000
25%,40.450000,95.800000,6.642500,8752.500000,-0.255000,75.700000,31.750000
50%,51.600000,101.900000,6.950000,9864.000000,0.080000,77.300000,34.200000
75%,61.000000,108.300000,7.995000,10660.500000,0.452500,78.350000,36.950000
max,76.000000,131.800000,10.080000,14044.000000,0.990000,80.000000,41.400000


In [125]:
df.count()

State                                   56
Diversity Index (%)                     51
Healthcare Cost Index (U.S. = 100)      51
Health Care as % of Household Income    50
Per-Capita Health Care Spending ($)     51
Health Score                            50
Life Expectancy (Years)                 51
Adult Obesity (%)                       55
dtype: int64

In [126]:
df.isnull().sum()

State                                   0
Diversity Index (%)                     5
Healthcare Cost Index (U.S. = 100)      5
Health Care as % of Household Income    6
Per-Capita Health Care Spending ($)     5
Health Score                            6
Life Expectancy (Years)                 5
Adult Obesity (%)                       1
dtype: int64

Filling NaN with Median for data estimation

In [127]:
numeric_cols = df.select_dtypes(include="number").columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

In [128]:
df.head()

,State,Diversity Index (%),Healthcare Cost Index (U.S. = 100),Health Care as % of Household Income,Per-Capita Health Care Spending ($),Health Score,Life Expectancy (Years),Adult Obesity (%)
0,Alabama,53.1,85.6,7.32,7980.0,-0.75,73.8,38.9
1,Alaska,62.8,130.7,10.08,14044.0,-0.13,75.8,34.0
2,Arizona,61.5,98.8,6.81,8556.0,-0.06,76.7,33.3
3,Arkansas,49.8,85.8,7.67,8562.0,-0.83,73.9,38.9
4,California,69.7,111.1,5.64,11054.0,0.10,79.3,29.1


In [130]:
df.to_csv('final_data(Filled NaN).csv', sep=',', index=False, encoding='utf-8')